In [1]:
import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
ngpu=1
ngf,nc = 3,3
ndf = 64

fsc_test={}

transform = transforms.Compose([
    transforms.Resize((400,400)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


#image="/kaggle/input/fake-car/WhatsApp Image 2024-11-16 at 6.36.05 PM.jpeg"
#image=cv2.cvtColor(cv2.resize(cv2.imread(image),(400,400)), cv2.COLOR_BGR2HSV)
#cv2.imwrite(f"/kaggle/working/image.jpg", image)

#image=transform(Image.open(f"/kaggle/working/image.jpg"))

for i in os.listdir('/kaggle/input/fake-scene-dataset/test'):
    fsc_test[i]=transform(Image.open(f'/kaggle/input/fake-scene-dataset/test/{i}'))

print(fsc_test["21.jpg"].shape)
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

torch.Size([3, 400, 400])


In [5]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 2),
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return torch.argmax(x, dim=1)

In [6]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
#EFF_NET
#EFF_NET.load_state_dict(torch.load("/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/0.00015554654237348586 87.0.pth",weights_only=False,map_location=torch.device('cpu')))
#EFF_NET.load_state_dict(torch.load(f"/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/0.00010106547415489331 29.pth", weights_only=False, map_location=torch.device('cpu')))


Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 154MB/s]


In [7]:
#EFF_NET(image.reshape((1, 3, 400, 400)).float().to(device)).cpu().detach().numpy()

In [8]:
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')
sub={}

for j in ['0.00010106547415489331 29.pth']:
    print(j)
    z_add,z_total=0,0
    EFF_NET.load_state_dict(torch.load(f"/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/{j}", 
                                       weights_only=False, map_location=torch.device('cpu')))
    for i in fsc_submission.index:
        img=fsc_test[i].reshape((1, 3, 400, 400)).float().to(device)
        #print(FSC_net(img))

        output = EFF_NET(img).cpu().detach().numpy()[0]
        if output==0:
            z_add+=1
        z_total+=1
        #fsc_submission.loc[i]['label']=output
        if i not in sub.keys():
            if output==0:
                sub[i]={0:1,
                       1:0}
            else:
                sub[i]={0:0,
                       1:1}
        else:
            if output==0:
                sub[i][0]+=1
            else:
                sub[i][1]+=1
    print(z_total,z_add)
    
for i in fsc_submission.index:
    sub[i]=list(sub[i].values())

0.00010106547415489331 29.pth
180 113


In [9]:
z_add,z_total=0,0

for i in fsc_submission.index:
    if sub[i][0]!=0:
        fsc_submission.loc[i]['label']=0
        z_add+=1
    else:
        fsc_submission.loc[i]['label']=1
    z_total+=1
print(z_total,z_add)

pandas.DataFrame(fsc_submission)

180 113


/tmp/ipykernel_17/91908440.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  fsc_submission.loc[i]['label']=0
/tmp/ipykernel_17/91908440.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting valu

,label
image,
102.jpg,0
108.jpg,0
109.jpg,0
111.jpg,0
121.jpg,0
...,...
899.jpg,0
91.jpg,0
94.jpg,0


In [10]:
fsc_submission['image']=fsc_submission.index
pandas.DataFrame(fsc_submission).to_csv(f"submission_15.csv", index=False)